# NB-11: VAE Decoder and Patchify Disambiguation

## Learning Objectives
1. Trace `Decoder3d`'s upsampling path as the mirror of `Encoder3d` (NB-10), with all shape transformations from latent to reconstructed video
2. Understand three key decoder differences from the encoder: (a) `in_dim` halving at levels 1-3, (b) extra ResBlock per level (`num_res_blocks + 1`), (c) channel halving in all upsample `Resample` modes
3. Distinguish VAE `patchify` (einops, no params, 5D output) from DiT patchify (Conv3d, learned params, 2D token sequence)

## Prerequisites
- **NB-09**: CausalConv3d, ResidualBlock, AttentionBlock (the building blocks used at every decoder level)
- **NB-10**: Encoder3d (decoder mirrors the encoder -- read to see the shape table format)
- **NB-07**: DiT patchify/unpatchify (for the disambiguation comparison in Section 3)

## Concept Map

```
Decoder3d Data Flow (dim=96, z_dim=16 production) -- MIRROR of Encoder3d
=========================================================================
[B, 16, T, H/8, W/8]  -- latent
       |
       v  conv1: CausalConv3d(z_dim=16 -> dims[0]=384)
[B, 384, T, H/8, W/8]
       |
       v  Middle: ResBlock + AttentionBlock + ResBlock  (BEFORE upsamples)
[B, 384, T, H/8, W/8]
       |
       v  Level 0: 3x ResBlock(384->384) + Resample(upsample3d) -> channels /2
[B, 192, T, H/4, W/4]      (temporal unchanged w/o feat_cache)
       |
       v  Level 1: 3x ResBlock(192->384) + Resample(upsample3d) -> channels /2  [in_dim halved]
[B, 192, T, H/2, W/2]
       |
       v  Level 2: 3x ResBlock(192->192) + Resample(upsample2d) -> channels /2  [in_dim halved]
[B, 96, T, H, W]
       |
       v  Level 3: 3x ResBlock(96->96)  (NO upsample -- last level)  [in_dim halved]
       |
       v  Head: RMS_norm + SiLU + CausalConv3d(96 -> 3)
[B, 3, T, H, W]  -- reconstructed video
```

In [ ]:
# Source: established in NB-01, adapted for wan_video_vae.py
import sys, importlib.util, pathlib, types

# Stub tqdm to suppress progress bars in notebook
_tqdm_stub = types.ModuleType('tqdm')
_tqdm_stub.tqdm = lambda x, **kw: x
sys.modules['tqdm'] = _tqdm_stub

# Walk up from notebook location to find the project root
_here = pathlib.Path().resolve()
PROJECT_ROOT = None
_candidate = _here
for _ in range(10):
    if (_candidate / 'diffsynth' / 'models' / 'wan_video_vae.py').exists():
        PROJECT_ROOT = _candidate
        break
    _parent = _candidate.parent
    if _parent == _candidate:
        break
    _candidate = _parent
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not find diffsynth/models/wan_video_vae.py')
print(f'Project root: {PROJECT_ROOT}')

# Load wan_video_vae.py directly via importlib
_vae_path = PROJECT_ROOT / 'diffsynth' / 'models' / 'wan_video_vae.py'
_spec = importlib.util.spec_from_file_location('diffsynth.models.wan_video_vae', _vae_path)
vae = importlib.util.module_from_spec(_spec)
sys.modules['diffsynth.models.wan_video_vae'] = vae
_spec.loader.exec_module(vae)

from diffsynth.models.wan_video_vae import (
    CausalConv3d, RMS_norm, ResidualBlock, AttentionBlock,
    Resample, Encoder3d, Decoder3d, VideoVAE_, patchify, unpatchify
)
import torch
from einops import rearrange
print('Setup complete.')

## 1. Decoder3d -- Mirroring the Encoder

`Decoder3d` mirrors `Encoder3d` in reverse: it takes a latent `[B, z_dim, T, H/8, W/8]` and reconstructs the video `[B, 3, T, H, W]`. The concept map above shows this is a symmetric reverse of NB-10's encoder diagram.

Three key differences from the encoder:

1. **Channel halving in `Resample`**: both `upsample2d` and `upsample3d` use `Conv2d(dim, dim//2, 3, padding=1)` -- the Conv2d **halves** output channels at every upsample step
2. **Extra ResBlock per level**: encoder uses `num_res_blocks` ResBlocks, decoder uses `num_res_blocks + 1` ResBlocks per level
3. **`in_dim` halving**: at levels 1, 2, and 3 (source: `wan_video_vae.py` line 770-771: `if i == 1 or i == 2 or i == 3: in_dim = in_dim // 2`), the effective input dim for ResBlocks is halved to compensate for the channel halving in the preceding `Resample`

| Sub-module | Type | Encoder Mirror? | Key Difference |
|-----------|------|----------------|----------------|
| `conv1` | `CausalConv3d(z_dim, dims[0], 3, p=1)` | Yes | Input is `z_dim` latent (not 3-ch RGB) |
| `middle` | `Sequential(ResBlock, AttnBlock, ResBlock)` | Yes -- identical | Applied **before** upsamples (encoder: after downsamples) |
| `upsamples` | `nn.Sequential(...)` | Mirror | `num_res_blocks + 1` ResBlocks per level; all Resample modes halve channels |
| `head` | `Sequential(RMS_norm, SiLU, CausalConv3d(dim->3))` | Yes | Output is 3-channel RGB |

**Source:** `diffsynth/models/wan_video_vae.py`, lines 736-838

In [ ]:
# Source: diffsynth/models/wan_video_vae.py, lines 736-838
# --- DEMO CONFIG (dim=8, matches NB-10's encoder for round-trip) ---
dec = Decoder3d(
    dim=8,                                    # vs 96 in production
    z_dim=4,                                  # vs 16 in production
    dim_mult=[1, 2, 4, 4],                    # SAME as production
    num_res_blocks=1,                         # vs 2 in production
    temperal_upsample=[True, True, False]     # reversed from encoder's temperal_downsample
)
# --- PRODUCTION CONFIG (annotation only) ---
# dec = Decoder3d(dim=96, z_dim=16, dim_mult=[1,2,4,4], num_res_blocks=2,
#                 temperal_upsample=[True, True, False])

print(f'conv1: {dec.conv1}')       # CausalConv3d(4, 32, kernel_size=3, padding=1)
print(f'upsamples: {len(dec.upsamples)} layers')
print(f'middle: {len(dec.middle)} layers')
print(f'head: {len(dec.head)} layers')
print()
print('upsamples breakdown:')
for i, m in enumerate(dec.upsamples):
    if hasattr(m, 'in_dim'):
        print(f'  [{i}] {type(m).__name__}({m.in_dim}->{m.out_dim})')
    else:
        print(f'  [{i}] {type(m).__name__}(mode={m.mode})')

### in_dim Halving -- The Decoder's Channel Compensation

The decoder computes `dims = [dim * u for u in [dim_mult[-1]] + dim_mult[::-1]]`. With `dim=8, dim_mult=[1,2,4,4]`: `dims = [32, 32, 32, 16, 8]`. At each level `i`, the ResBlocks process tensors from `dims[i]` to `dims[i+1]`.

**The problem**: After Level 0's `Resample(upsample3d)`, the output channels are `dims[0] // 2 = 16` (because both upsample modes halve channels). But `dims[1] = 32`. Without adjustment, Level 1's ResBlocks would expect a 32-channel input but receive 16.

**The fix** (source line 770-771): `if i == 1 or i == 2 or i == 3: in_dim = in_dim // 2`. This halves the expected input dim for ResBlocks at those levels, matching the actual channels coming from the previous `Resample`.

In [ ]:
# Source: diffsynth/models/wan_video_vae.py, lines 768-782
# Decoder3d.__init__ loops over dims pairs and applies in_dim halving:
dim = 8
dim_mult = [1, 2, 4, 4]
dims = [dim * u for u in [dim_mult[-1]] + dim_mult[::-1]]  # [32, 32, 32, 16, 8]
# Note: dims starts with dim_mult[-1]*dim, then reverses dim_mult

print('Decoder level-by-level in_dim computation:')
print(f'dims (base): {dims}')
for i, (in_dim_orig, out_dim) in enumerate(zip(dims[:-1], dims[1:])):
    in_dim = in_dim_orig
    if i == 1 or i == 2 or i == 3:
        in_dim = in_dim // 2  # compensate for Resample channel halving
    upsample = i < len(dim_mult) - 1  # not last level
    print(f'  Level {i}: orig_in_dim={in_dim_orig}, used_in_dim={in_dim} (halved={i in [1,2,3]}), out_dim={out_dim}, has_resample={upsample}')

# Level 0: in_dim=32, out_dim=32 (no halving), Resample(upsample3d) -> output 16ch
# Level 1: in_dim=32->16 (halved), out_dim=32, Resample(upsample3d) -> output 16ch
# Level 2: in_dim=32->16 (halved), out_dim=16, Resample(upsample2d) -> output 8ch
# Level 3: in_dim=16->8 (halved), out_dim=8, NO Resample (last level)

## 2. Decoder Shape Trace

The decoder shape trace is the reverse of NB-10's encoder table. Starting from latent `[1, 4, 5, 2, 2]` and reconstructing to `[1, 3, 5, 16, 16]`.

**Demo Shape Table (dim=8, z_dim=4, input [1,4,5,2,2]):**

| Stage | Tensor Shape | Operation |
|-------|-------------|----------|
| Input z | `[1, 4, 5, 2, 2]` | Latent from encoder |
| After conv1 | `[1, 32, 5, 2, 2]` | CausalConv3d(4->32, k=3, p=1) |
| After Middle | `[1, 32, 5, 2, 2]` | ResBlock + AttentionBlock + ResBlock |
| After L0 ResBlocks | `[1, 32, 5, 2, 2]` | 2x ResidualBlock(32->32) |
| After L0 Resample | `[1, 16, 5, 4, 4]` | upsample3d: spatial x2, channels /2 (32->16) |
| After L1 ResBlocks | `[1, 32, 5, 4, 4]` | 2x ResidualBlock(16->32) [in_dim halved: 32//2=16] |
| After L1 Resample | `[1, 16, 5, 8, 8]` | upsample3d: spatial x2, channels /2 (32->16) |
| After L2 ResBlocks | `[1, 16, 5, 8, 8]` | 2x ResidualBlock(16->16) [in_dim halved: 32//2=16] |
| After L2 Resample | `[1, 8, 5, 16, 16]` | upsample2d: spatial x2, channels /2 (16->8) |
| After L3 ResBlocks | `[1, 8, 5, 16, 16]` | 2x ResidualBlock(8->8) [in_dim halved: 16//2=8] |
| After Head | `[1, 3, 5, 16, 16]` | RMS_norm + SiLU + CausalConv3d(8->3) |

Note: `num_res_blocks + 1 = 2` ResBlocks per level in demo (production: 3).

**Key observation**: All `Resample` upsample modes (both `upsample2d` and `upsample3d`) use `Conv2d(dim, dim//2, 3, padding=1)` -- they ALL halve channels. This is why `in_dim` halving is needed at levels 1, 2, 3.

In [ ]:
# Encode-decode round-trip using NB-10's encoder
# Encoder uses z_dim=8 to produce 8-channel output for mu/log_var split
enc = Encoder3d(dim=8, z_dim=8, dim_mult=[1,2,4,4], num_res_blocks=1,
                temperal_downsample=[False, True, True])
x_video = torch.randn(1, 3, 5, 16, 16)  # [B, C=3, T=5, H=16, W=16]
with torch.no_grad():
    enc_out = enc(x_video)       # [1, 8, 5, 2, 2] (z_dim=8)

# Split and reparameterize
mu, log_var = enc_out.chunk(2, dim=1)
std = torch.exp(0.5 * log_var)
z = mu + torch.randn_like(std) * std   # [1, 4, 5, 2, 2]

# Decode
with torch.no_grad():
    x_recon = dec(z)             # [1, 3, 5, 16, 16]

print(f'Input video:     {list(x_video.shape)}')   # [1, 3, 5, 16, 16]
print(f'Latent z:        {list(z.shape)}')          # [1, 4, 5, 2, 2]
print(f'Reconstructed:   {list(x_recon.shape)}')    # [1, 3, 5, 16, 16]
assert x_recon.shape == x_video.shape, f'Round-trip shape mismatch: {x_recon.shape} != {x_video.shape}'
print(f'\nEncode-Decode round-trip: shape PRESERVED')
print(f'Spatial:  {x_video.shape[3]}x{x_video.shape[4]} -> {z.shape[3]}x{z.shape[4]} -> {x_recon.shape[3]}x{x_recon.shape[4]}')
print(f'Temporal: {x_video.shape[2]} -> {z.shape[2]} -> {x_recon.shape[2]} (unchanged w/o feat_cache)')

### Upsample Resample Internals

Both `upsample2d` and `upsample3d` use identical spatial upsampling: `Upsample(scale_factor=(2,2))` followed by `Conv2d(dim, dim//2, 3, padding=1)`. The channel halving in the Conv2d is why the decoder needs `in_dim` compensation at levels 1-3.

The difference between `upsample2d` and `upsample3d` is temporal: `upsample3d` also has a `time_conv = CausalConv3d(dim, dim*2, (3,1,1))` for temporal upsampling (doubles temporal frames via interleaving), but like `downsample3d`, it only runs with `feat_cache`.

In [ ]:
# Source: diffsynth/models/wan_video_vae.py, lines 82-196
rs_up2 = Resample(8, 'upsample2d')
rs_up3 = Resample(16, 'upsample3d')

print('upsample2d internals:')
print(f'  resample: {rs_up2.resample}')
# Sequential(Upsample(scale_factor=(2,2)), Conv2d(8, 4, 3, padding=1))
# Note: channels HALVED (8 -> 4) for upsample2d!

print('\nupsample3d internals:')
print(f'  resample (spatial): {rs_up3.resample}')
# Sequential(Upsample(scale_factor=(2,2)), Conv2d(16, 8, 3, padding=1))
# Note: channels HALVED (16 -> 8) for upsample3d!
print(f'  time_conv: {rs_up3.time_conv}')
# CausalConv3d(16, 32, (3,1,1)) -- temporal upsampling via channel expansion+interleave

x_up2 = torch.randn(1, 8, 4, 4, 4)    # [B, C=8, T=4, H=4, W=4]
x_up3 = torch.randn(1, 16, 4, 4, 4)   # [B, C=16, T=4, H=4, W=4]
with torch.no_grad():
    out_up2 = rs_up2(x_up2)
    out_up3 = rs_up3(x_up3)

print(f'\nupsample2d: {list(x_up2.shape)} -> {list(out_up2.shape)}')
# [1, 8, 4, 4, 4] -> [1, 4, 4, 8, 8]  spatial x2, channels HALVED (8->4)
print(f'upsample3d: {list(x_up3.shape)} -> {list(out_up3.shape)}')
# [1, 16, 4, 4, 4] -> [1, 8, 4, 8, 8]  spatial x2, channels HALVED (16->8)
assert out_up2.shape == (1, 4, 4, 8, 8), f'Expected (1,4,4,8,8), got {out_up2.shape}'
assert out_up3.shape == (1, 8, 4, 8, 8), f'Expected (1,8,4,8,8), got {out_up3.shape}'
print('\nBoth upsample modes HALVE channels -- this is WHY in_dim halving is needed at levels 1, 2, 3')

## 3. VAE Patchify vs DiT Patchify

The name "patchify" appears in both `wan_video_vae.py` and as a method on `WanModel` (from NB-07), but they are fundamentally different operations with different purposes, implementations, and outputs.

| Property | VAE `patchify` (`wan_video_vae.py:199`) | DiT patchify (`WanModel.patch_embedding`, NB-07) |
|----------|-----------------------------------------|--------------------------------------------------|
| Implementation | `einops.rearrange` | `nn.Conv3d` + `rearrange` |
| Learned parameters | **None** (deterministic rearrangement) | **Yes** (~296K Conv3d weights + bias) |
| Input | `[B, C, F, H, W]` (5D video) | `[B, 48, F, H, W]` (48-ch: 16 noise+16 ctrl+16 ref) |
| Output | `[B, C*4, F, H/2, W/2]` (still 5D video) | `[B, F*h*w, 1536]` (2D token sequence) |
| Temporal | Unchanged | Unchanged (patch_size[0]=1) |
| Spatial | H,W halved; spatial pixels -> channels | H,W halved (Conv stride=2); flattened to tokens |
| Purpose | Channel subdivision for `VideoVAE38_` (3.8B model) | Input projection for DiT blocks (NB-07) |
| Exact inverse? | **Yes** -- `unpatchify` is exact | **No** -- Conv3d is not exactly invertible |
| Called by | `VideoVAE38_.encode/decode` only | `WanModel.forward` |

**Note**: The 1.3B `VideoVAE_` model used in production **does NOT call `patchify`**. Only the 3.8B `VideoVAE38_` uses it. Both models share the same `patchify` function definition in `wan_video_vae.py`.

### ASCII Diagram: VAE patchify vs DiT patchify

```
VAE patchify vs DiT patchify
=============================
VAE patchify (einops, no params):
  [B, 16, F, H, W] -> rearrange 'b c f (h q) (w r) -> b (c r q) f h w'
  -> [B, 64, F, H/2, W/2]    Still a 5D video tensor!
  Spatial pixels -> channels  (2x2 spatial patch becomes 4 extra channels)
  Exact inverse: unpatchify reverses the rearrange exactly.

DiT patchify (Conv3d + rearrange, has params):
  [B, 48, F, H, W] -> Conv3d(48, 1536, k=(1,2,2), s=(1,2,2)) -> [B, 1536, F, H/2, W/2]
  -> rearrange 'b c f h w -> b (f h w) c' -> [B, F*H/2*W/2, 1536]
  Spatial patches + channels -> flat TOKEN SEQUENCE for DiT attention
  No exact inverse: Conv3d projection loses information.

See NB-07 for the full DiT patchify/unpatchify implementation.
```

In [ ]:
# Source: diffsynth/models/wan_video_vae.py, lines 199-211
# VAE patchify: einops rearrange -- NO learned parameters
# Used by VideoVAE38_ (3.8B model) only; VideoVAE_ (1.3B) does NOT use patchify
x_5d = torch.randn(1, 16, 4, 8, 8)  # [B, C=16, F=4, H=8, W=8]
vae_patched = patchify(x_5d, patch_size=2)
# rearrange 'b c f (h q) (w r) -> b (c r q) f h w', q=2, r=2
print(f'VAE patchify: {list(x_5d.shape)} -> {list(vae_patched.shape)}')
# [1, 16, 4, 8, 8] -> [1, 64, 4, 4, 4]
assert vae_patched.shape == (1, 64, 4, 4, 4), f'Expected (1,64,4,4,4), got {vae_patched.shape}'
print(f'  Channels: {x_5d.shape[1]} -> {vae_patched.shape[1]} (x{vae_patched.shape[1]//x_5d.shape[1]} = patch_size^2={2**2})')
print(f'  Spatial:  {x_5d.shape[3]}x{x_5d.shape[4]} -> {vae_patched.shape[3]}x{vae_patched.shape[4]} (/2)')
print(f'  Temporal: {x_5d.shape[2]} -> {vae_patched.shape[2]} (UNCHANGED)')
print(f'  Parameters: NONE (pure channel rearrangement)')

In [ ]:
# VAE unpatchify reverses patchify EXACTLY (both are einops rearrange, no params)
recovered = unpatchify(vae_patched, patch_size=2)
assert recovered.shape == x_5d.shape, f'Shape mismatch: {recovered.shape} != {x_5d.shape}'
assert torch.allclose(recovered, x_5d), 'Values should be EXACTLY preserved (einops only)'
print(f'Round-trip: {list(x_5d.shape)} -> patchify -> {list(vae_patched.shape)} -> unpatchify -> {list(recovered.shape)}')
print('VAE patchify/unpatchify: EXACT value round-trip (no learned params)')
print()
print('Contrast: DiT patchify uses Conv3d (learned weights) -- no exact inverse exists.')
print('See NB-07 for the full DiT patchify/unpatchify implementation.')

In [ ]:
# Parameter comparison: VAE patchify vs DiT patchify
print('VAE patchify parameters: 0 (pure einops rearrange)')

# DiT patchify (from NB-07): Conv3d(48, 1536, kernel_size=(1,2,2), stride=(1,2,2))
# Parameters: weight [1536, 48, 1, 2, 2] + bias [1536]
weight_params = 1536 * 48 * 1 * 2 * 2   # 294,912
bias_params = 1536
total = weight_params + bias_params       # 296,448
print(f'DiT patch_embedding parameters: {total:,}')
print(f'  Conv3d weight: {weight_params:,} = 1536 x 48 x 1 x 2 x 2')
print(f'  Conv3d bias:   {bias_params:,}')

print()
print('Summary of differences:')
print('  VAE patchify: einops only, 0 params, output is 5D video [B, C*4, F, H/2, W/2]')
print(f'  DiT patchify: Conv3d, {total:,} params, output is 2D tokens [B, F*h*w, 1536]')

## Summary

### Key Takeaways
- **`Decoder3d`** mirrors `Encoder3d` in reverse: latent `[B, z_dim, T, H/8, W/8]` -> upsampled `[B, 3, T, H, W]` video
- **Three key decoder differences**: (1) both `upsample2d` and `upsample3d` halve channels via `Conv2d(dim, dim//2)`, (2) `num_res_blocks + 1` ResBlocks per level, (3) `in_dim //= 2` at levels 1-3 to compensate for channel halving
- **`dims` computation**: encoder uses `[dim * u for u in [1] + dim_mult]`; decoder uses `[dim * u for u in [dim_mult[-1]] + dim_mult[::-1]]` -- reversed, starting from the largest dim
- **VAE `patchify`**: `einops.rearrange` channel rearrangement (`[B,C,F,H,W] -> [B,C*4,F,H/2,W/2]`), zero learned parameters, EXACT inverse via `unpatchify`. Used by `VideoVAE38_` (3.8B model) only.
- **DiT `patchify`** (NB-07): `Conv3d` learned projection (`[B,48,F,H,W] -> [B,F*h*w,1536]` token sequence), ~296K parameters. No exact inverse.

### Source References

| Symbol | Location |
|--------|----------|
| `Decoder3d` | `diffsynth/models/wan_video_vae.py`, line 736 |
| `patchify` | `diffsynth/models/wan_video_vae.py`, line 199 |
| `unpatchify` | `diffsynth/models/wan_video_vae.py`, line 213 |
| DiT `patch_embedding` | `wan_video_dit.py` `WanModel`, see NB-07 |
| `Encoder3d` | NB-10, `wan_video_vae.py`, line 517 |

## Exercises

### Exercise 1 -- Decoder with different z_dim
Create `Decoder3d(dim=8, z_dim=8, dim_mult=[1,2,4,4], num_res_blocks=1, temperal_upsample=[True,True,False])`. How does `conv1` change? What's the new latent input shape needed (i.e. what channel count does the decoder expect)? Run a forward pass with the correct input shape to verify.

### Exercise 2 -- Patchify with patch_size=4
Call `patchify(x, patch_size=4)` on a `[1, 16, 4, 16, 16]` tensor. What's the output shape? How many channels? (Hint: channels multiply by `patch_size^2`.) Verify with `unpatchify` round-trip that `torch.allclose` returns `True`.

### Exercise 3 -- Count total encoder+decoder parameters
Instantiate both `Encoder3d(dim=8, z_dim=8, dim_mult=[1,2,4,4], num_res_blocks=1, temperal_downsample=[False,True,True])` and `Decoder3d(dim=8, z_dim=4, dim_mult=[1,2,4,4], num_res_blocks=1, temperal_upsample=[True,True,False])`. Use `sum(p.numel() for p in model.parameters())` for each. Which has more parameters and why? (Hint: extra ResBlock per level in decoder)